In [1]:
import pandas as pd
import numpy as np
import pickle

In [37]:
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression

## Notebook to evaluate embedding methods (extrinsic) with reference data

Steps:
1. Load NPKG version 3 (hashed with weights)
2. Load embeddings (after DGLKE modeling and processing)
3. Load reference data
4. Feature extraction -- edge feature methods 
5. Cross validation
6. Classifier setup
7. Evaluate with metrics
8. Visualizations -- separate notebook 

In [3]:
EMBED_PATH = 'embeddings/'
DATA_PATH = '../data/'

### Load embeddings

In [4]:
model_name = 'TransE_l1'
embed_file_name = 'npkg_'+model_name+'_embeddings.pickle'
with open(EMBED_PATH+embed_file_name, 'rb') as filep:
    transe_embedding_dict = pickle.load(filep)
transe_embedding_dict.keys()

dict_keys(['embedding_method_name', 'node_embeddings', 'edge_embeddings', 'node_type_embeddings', 'edge_type_embeddings'])

In [ ]:
model_name = 'ComplEx'
embed_file_name = 'npkg_'+model_name+'_embeddings.pickle'
with open(EMBED_PATH+embed_file_name, 'rb') as filep:
    complex_embedding_dict = pickle.load(filep)

In [ ]:
model_name = 'DistMult'
embed_file_name = 'npkg_'+model_name+'_embeddings.pickle'
with open(EMBED_PATH+embed_file_name, 'rb') as filep:
    distmult_embedding_dict = pickle.load(filep)

### Load reference data
* convert URIs to CURIEs
* save edges

In [5]:
filename = 'reference-data/reference_data_curie_edges_20240725.tsv'

In [6]:
refdf = pd.read_csv(DATA_PATH+filename, sep='\t')
refdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1364 entries, 0 to 1363
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   source            1364 non-null   object
 1   target            1364 non-null   object
 2   interaction_type  1364 non-null   object
dtypes: object(3)
memory usage: 32.1+ KB


In [7]:
le = LabelEncoder()
refdf['interaction_type_encoded'] = le.fit_transform(refdf['interaction_type'])
refdf.head()

,source,target,interaction_type,interaction_type_encoded
0,chebi:16118,chebi:16469,positive,2
1,chebi:4806,chebi:16469,positive,2
2,chebi:69919,chebi:791,positive,2
3,napdi_srs_imports:hydrastis_canadensis,chebi:791,positive,2
4,chebi:31018,chebi:17224,negative,1


### Edge Prediction Evaluation

1. Feature extraction 
2. Baseline with logistic regression
3. Multilayer perceptron

In [8]:
def get_embeddings_df(embedding_dict):
    node_embeddings_df = embedding_dict['node_embeddings'][0]
    edge_type_embeddings_df = embedding_dict['edge_type_embeddings'][0]
    return node_embeddings_df, edge_type_embeddings_df

In [31]:
def extract_features(edges, node_embeddings_df, edge_type_embeddings_df, edge_feature_method='concat'):
    features = []
    for u, v, edge_type in edges:
        try:
            node_u_embedding = node_embeddings_df.loc[u].values
            node_v_embedding = node_embeddings_df.loc[v].values
            #edge_type_embedding = edge_type_embeddings_df.loc[edge_type].values
            if edge_feature_method == 'concat':
                node_feature = np.concatenate([node_u_embedding, node_v_embedding])
            elif edge_feature_method == 'hadamard':
                node_feature = np.multiply(node_u_embedding,node_v_embedding)
            features.append(node_feature)
        except KeyError:
            continue
    return np.array(features)

In [10]:
edges = refdf[['source', 'target', 'interaction_type_encoded']].values

In [ ]:
##Run below for each embedding model

In [13]:
embedding_model_name = 'TransE'

In [14]:
k = 5  # Number of folds
kf = KFold(n_splits=k, shuffle=True, random_state=42)

In [15]:
accuracies = []
precisions = []
recalls = []
f1s = []
aucs = []

In [22]:
if embedding_model_name == 'TransE':
    embedding_dict = transe_embedding_dict
elif embedding_model_name == 'ComplEx':
    embedding_dict = complex_embedding_dict
elif embedding_model_name == 'DistMult':
    embedding_dict = distmult_embedding_dict
node_embeddings_df, edge_type_embeddings_df = get_embeddings_df(embedding_dict)

In [32]:
X = extract_features(edges, node_embeddings_df, edge_type_embeddings_df, edge_feature_method='concat')
y = refdf['interaction_type_encoded'].values
#Handle imbalanced data
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

In [52]:
accuracies = []
precisions = []
recalls = []
f1s = []
aucs = []
refdf_predict = refdf.copy()
refdf_predict['lr_edge_proba'] = np.nan

In [53]:
classifier_lr = LogisticRegression(class_weight=class_weight_dict, max_iter=1000)

In [56]:
for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    classifier_lr.fit(X_train, y_train)
    y_pred = classifier_lr.predict(X_test)
    y_proba = classifier_lr.predict_proba(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred, average='weighted'))
    recalls.append(recall_score(y_test, y_pred, average='weighted'))
    f1s.append(f1_score(y_test, y_pred, average='weighted'))
    aucs.append(roc_auc_score(y_test, y_proba, multi_class='ovr'))
    refdf_predict.loc[test_index, 'lr_edge_proba'] = y_proba[:, 1]

In [57]:
print("Accuracy: {:.2f} ± {:.2f}".format(np.mean(accuracies), np.std(accuracies)))
print("Precision: {:.2f} ± {:.2f}".format(np.mean(precisions), np.std(precisions)))
print("Recall: {:.2f} ± {:.2f}".format(np.mean(recalls), np.std(recalls)))
print("F1 Score: {:.2f} ± {:.2f}".format(np.mean(f1s), np.std(f1s)))
print("AUC: {:.2f} ± {:.2f}".format(np.mean(aucs), np.std(aucs)))

Accuracy: 0.65 ± 0.02
Precision: 0.83 ± 0.03
Recall: 0.65 ± 0.02
F1 Score: 0.71 ± 0.02
AUC: 0.77 ± 0.03


In [58]:
refdf_predict.head()

,source,target,interaction_type,interaction_type_encoded,lr_edge_proba
0,chebi:16118,chebi:16469,positive,2,0.275552
1,chebi:4806,chebi:16469,positive,2,0.213551
2,chebi:69919,chebi:791,positive,2,0.367630
3,napdi_srs_imports:hydrastis_canadensis,chebi:791,positive,2,0.320826
4,chebi:31018,chebi:17224,negative,1,0.315640


In [ ]:
classifier_mlp = MLPClassifier(
    hidden_layer_sizes=(100,), 
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size='auto',
    learning_rate="constant",
    learning_rate_init=0.001,
    power_t=0.5,
    max_iter=200,
    validation_fraction=0.1,
    shuffle=True,
    tol=1e-4,
    random_state=42, 
    class_weight=class_weight_dict
)